# Chicago Crime - Advanced Modeling & Experiments

**Extended experiments**: Additional models (SVM, KNN, Naive Bayes, ExtraTrees), data preparation experiments, LightGBM v2 pipeline, extended hyperparameter tuning.

Run this notebook after `04_data_modeling.ipynb` for baseline context, or run standalone (loads data and prepares features).

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/crimes_cleaned.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (2441506, 28)


,ID,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,...,DayOfWeek,DayOfWeekName,Hour,Quarter,WeekOfYear,IsWeekend,Lat_bin,Lon_bin,HourBin,Crime_Category
0,14135339,2026-03-13 00:00:00,075XX S KINGSTON AVE,0760,BURGLARY,BURGLARY FROM MOTOR VEHICLE,STREET,0,0,421,...,4,Friday,0,1,11,False,2087,-4379,0,Theft
1,14135179,2026-03-13 00:00:00,050XX N MARINE DR,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE - GARAGE,0,0,2024,...,4,Friday,0,1,11,False,2098,-4383,0,Property
2,14138214,2026-03-13 00:00:00,075XX S STONY ISLAND AVE,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,HOSPITAL BUILDING / GROUNDS,0,0,411,...,4,Friday,0,1,11,False,2087,-4380,0,Violent


## 1. Ensure Modeling Columns Exist

Add Lat_bin, Lon_bin, HourBin, Crime_Category if not present (for compatibility with older cleaned data).

In [11]:
if 'Lat_bin' not in df.columns:
    df['Lat_bin'] = (df['Latitude'] // 0.02).astype(int)
if 'Lon_bin' not in df.columns:
    df['Lon_bin'] = (df['Longitude'] // 0.02).astype(int)
if 'HourBin' not in df.columns:
    def get_hour_bin(h):
        if 0 <= h < 4: return 0
        if 4 <= h < 8: return 1
        if 8 <= h < 12: return 2
        if 12 <= h < 16: return 3
        if 16 <= h < 20: return 4
        return 5
    df['HourBin'] = df['Hour'].apply(get_hour_bin)
if 'Crime_Category' not in df.columns:
    CRIME_MAP = {'Violent': ['BATTERY','ASSAULT','ROBBERY','CRIMINAL SEXUAL ASSAULT','HOMICIDE','KIDNAPPING'],
                 'Theft': ['THEFT','MOTOR VEHICLE THEFT','BURGLARY'], 'Property': ['CRIMINAL DAMAGE','CRIMINAL TRESPASS','ARSON'],
                 'Narcotics': ['NARCOTICS','OTHER NARCOTIC VIOLATION'], 'Other': []}
    def map_cat(pt):
        for c, t in CRIME_MAP.items():
            if pt in t: return c
        return 'Other'
    df['Crime_Category'] = df['Primary Type'].apply(map_cat)

top_loc = df['Location Description'].value_counts().head(20).index.tolist()
df['Location_enc'] = df['Location Description'].apply(lambda x: x if x in top_loc else 'OTHER')
df['Location_enc'] = LabelEncoder().fit_transform(df['Location_enc'].astype(str))

# Sample for faster training
SAMPLE_SIZE = 200000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42)
print("Sample size:", len(df))

Sample size: 200000


## 2. Prepare Features and Targets (Aligned with Predictions)

**Task 1 - Crime Category (5 classes)**: Beat, Lat_bin, Lon_bin, HourBin, DayOfWeek, Month, Location_enc, Domestic

**Task 2 - Beat (top 25 for balance)**: Lat_bin, Lon_bin, Crime_Category, HourBin, DayOfWeek, Month, Location_enc, Community_Area

**Task 3 - HourBin (6 classes)**: Beat, Crime_Category, DayOfWeek, Month, Location_enc, IsWeekend

In [12]:
df['Community_Area'] = df['Community Area'].fillna(-1).astype(int)

# Task 1: Crime Category
feat_crime = ['Beat', 'Lat_bin', 'Lon_bin', 'HourBin', 'DayOfWeek', 'Month', 'Location_enc', 'Domestic']
X1 = df[feat_crime].astype(float)
y1 = df['Crime_Category']

# Task 2: Top 25 Beats
top_beats = df['Beat'].value_counts().head(25).index.tolist()
mask_b = df['Beat'].isin(top_beats)
feat_beat = ['Lat_bin', 'Lon_bin', 'HourBin', 'DayOfWeek', 'Month', 'Location_enc', 'Community_Area']
df['Crime_Cat_enc'] = LabelEncoder().fit_transform(df['Crime_Category'])
feat_beat2 = ['Lat_bin', 'Lon_bin', 'Crime_Cat_enc', 'HourBin', 'DayOfWeek', 'Month', 'Location_enc', 'Community_Area']
X2 = df.loc[mask_b, feat_beat2].astype(float)
y2 = df.loc[mask_b, 'Beat'].astype(str)

# Task 3: HourBin
feat_hour = ['Beat', 'Crime_Cat_enc', 'DayOfWeek', 'Month', 'Location_enc', 'IsWeekend']
X3 = df[feat_hour].astype(float)
X3['IsWeekend'] = X3['IsWeekend'].astype(int)
y3 = df['HourBin']

print("Task 1 (Crime Cat):", X1.shape, y1.value_counts().to_dict())
print("Task 2 (Beat):", X2.shape)
print("Task 3 (HourBin):", X3.shape)

Task 1 (Crime Cat): (200000, 8) {'Theft': 65691, 'Violent': 62847, 'Other': 37517, 'Property': 27019, 'Narcotics': 6926}
Task 2 (Beat): (32984, 8)
Task 3 (HourBin): (200000, 6)


## 1. Experiment Log

Initialize the experiment log for advanced models. All experiments append to `EXPERIMENT_LOG` via `log_experiment()`.

In [13]:
EXPERIMENT_LOG = []

def log_experiment(task, model_name, params, accuracy, f1, notes=""):
    EXPERIMENT_LOG.append({
        'task': task, 'model': model_name, 'params': str(params),
        'accuracy': round(float(accuracy), 4), 'f1_weighted': round(float(f1), 4), 'notes': notes
    })


## 9. Additional Models for Crime_Category and HourBin

Try SVM, KNN, Naive Bayes, ExtraTrees on Crime_Category and HourBin only (Beat already meets target).

In [5]:
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import ExtraTreesClassifier

def run_alt_models(X, y, task_name, scaler=None):
    """Run SVM, KNN, Naive Bayes, ExtraTrees on Crime_Category or HourBin."""
    class_counts = y.value_counts()
    valid = class_counts[class_counts >= 2].index
    m = y.isin(valid)
    Xf, yf = X[m].reset_index(drop=True), y[m].reset_index(drop=True)
    X_tr, X_te, y_tr, y_te = train_test_split(Xf, yf, test_size=0.2, random_state=42, stratify=yf)
    if scaler is None:
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)
    else:
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

    # SVM_rbf is slow on large data; use sample for it
    if len(X_tr_s) > 50000:
        idx = np.random.RandomState(42).choice(len(X_tr_s), 30000, replace=False)
        X_tr_svm, y_tr_svm = X_tr_s[idx], y_tr.iloc[idx]
    else:
        X_tr_svm, y_tr_svm = X_tr_s, y_tr

    models = {
        'SVM_linear': LinearSVC(max_iter=2000, random_state=42, dual='auto'),
        'SVM_rbf': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=15, weights='distance', n_jobs=-1),
        'NaiveBayes': GaussianNB(),
        'ExtraTrees': ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    }
    print(f"\n--- {task_name} (Additional Models) ---")
    for name, mdl in models.items():
        X_fit, y_fit = (X_tr_svm, y_tr_svm) if name == 'SVM_rbf' and len(X_tr_s) > 50000 else (X_tr_s, y_tr)
        mdl.fit(X_fit, y_fit)
        pred = mdl.predict(X_te_s)
        acc = accuracy_score(y_te, pred)
        f1 = f1_score(y_te, pred, average='weighted')
        log_experiment(task_name, name, {}, acc, f1, 'Alt model')
        print(f"  {name}: Acc={acc:.4f} F1={f1:.4f}")
    return scaler

_ = run_alt_models(X1, y1, 'Crime_Category')
_ = run_alt_models(X3, y3, 'HourBin')


--- Crime_Category (Additional Models) ---
  SVM_linear: Acc=0.4425 F1=0.3420


KeyboardInterrupt: 

## 10. Data Preparation Experiments for Crime_Category and HourBin

Try alternate feature sets and target definitions to improve accuracy.

In [39]:
# --- Crime_Category: Alternate feature set ---
# Add: raw Hour (timing signal), Quarter, more Location categories (top 30), Community_Area
top_loc_30 = df['Location Description'].value_counts().head(30).index.tolist()
df['Location_enc_30'] = LabelEncoder().fit_transform(
    df['Location Description'].apply(lambda x: x if x in top_loc_30 else 'OTHER').astype(str)
)

feat_crime_alt = ['Beat', 'Lat_bin', 'Lon_bin', 'Hour', 'HourBin', 'DayOfWeek', 'Month', 'Quarter',
                  'Location_enc_30', 'Domestic', 'Community_Area', 'IsWeekend']
X1_alt = df[feat_crime_alt].astype(float)
X1_alt['IsWeekend'] = X1_alt['IsWeekend'].astype(int)
y1_alt = df['Crime_Category']

print("Crime_Category - Alternate features:", feat_crime_alt)
print("Running best models (GB, ExtraTrees, SVM_linear) on alternate features...")

Crime_Category - Alternate features: ['Beat', 'Lat_bin', 'Lon_bin', 'Hour', 'HourBin', 'DayOfWeek', 'Month', 'Quarter', 'Location_enc_30', 'Domestic', 'Community_Area', 'IsWeekend']
Running best models (GB, ExtraTrees, SVM_linear) on alternate features...


In [40]:
for name, mdl in [
    ('GB', GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('ExtraTrees', ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('SVM_linear', LinearSVC(max_iter=2000, random_state=42, dual='auto'))
]:
    X_tr, X_te, y_tr, y_te = train_test_split(X1_alt, y1_alt, test_size=0.2, random_state=42, stratify=y1_alt)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    mdl.fit(X_tr_s, y_tr)
    pred = mdl.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='weighted')
    log_experiment('Crime_Category', f'{name}_alt_feat', {}, acc, f1, 'Alt features')
    print(f"  {name} (alt features): Acc={acc:.4f} F1={f1:.4f}")

  GB (alt features): Acc=0.5057 F1=0.4562
  ExtraTrees (alt features): Acc=0.4399 F1=0.4212
  SVM_linear (alt features): Acc=0.4429 F1=0.3428


In [37]:
# --- HourBin: Alternate feature set (add spatial features) ---
feat_hour_alt = ['Beat', 'Crime_Cat_enc', 'DayOfWeek', 'Month', 'Location_enc', 'IsWeekend',
                 'Lat_bin', 'Lon_bin', 'Community_Area']
X3_alt = df[feat_hour_alt].astype(float)
X3_alt['IsWeekend'] = X3_alt['IsWeekend'].astype(int)
y3_alt = df['HourBin']

print("HourBin - Alternate features (add Lat_bin, Lon_bin, Community_Area):", feat_hour_alt)

HourBin - Alternate features (add Lat_bin, Lon_bin, Community_Area): ['Beat', 'Crime_Cat_enc', 'DayOfWeek', 'Month', 'Location_enc', 'IsWeekend', 'Lat_bin', 'Lon_bin', 'Community_Area']


In [41]:
for name, mdl in [
    ('GB', GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('ExtraTrees', ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('SVM_linear', LinearSVC(max_iter=2000, random_state=42, dual='auto'))
]:
    X_tr, X_te, y_tr, y_te = train_test_split(X3_alt, y3_alt, test_size=0.2, random_state=42, stratify=y3_alt)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    mdl.fit(X_tr_s, y_tr)
    pred = mdl.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='weighted')
    log_experiment('HourBin', f'{name}_alt_feat', {}, acc, f1, 'Alt features')
    print(f"  {name} (alt features): Acc={acc:.4f} F1={f1:.4f}")

  GB (alt features): Acc=0.2525 F1=0.2265
  ExtraTrees (alt features): Acc=0.2081 F1=0.2067
  SVM_linear (alt features): Acc=0.2221 F1=0.1600


In [42]:
# --- HourBin: Try 4 bins instead of 6 (Morning 0-6, Day 6-12, Evening 12-18, Night 18-24) ---
def hour_to_4bin(h):
    if 0 <= h < 6: return 0   # Night/early morning
    if 6 <= h < 12: return 1  # Morning
    if 12 <= h < 18: return 2  # Afternoon
    return 3  # Evening

df['HourBin4'] = df['Hour'].apply(hour_to_4bin)
y3_4bin = df['HourBin4']

print("HourBin4 (4 classes): random baseline 25%. Running models...")

HourBin4 (4 classes): random baseline 25%. Running models...


In [43]:
for name, mdl in [
    ('GB', GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('ExtraTrees', ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('SVM_linear', LinearSVC(max_iter=2000, random_state=42, dual='auto'))
]:
    X_tr, X_te, y_tr, y_te = train_test_split(X3_alt, y3_4bin, test_size=0.2, random_state=42, stratify=y3_4bin)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    mdl.fit(X_tr_s, y_tr)
    pred = mdl.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='weighted')
    log_experiment('HourBin4', f'{name}', {}, acc, f1, '4 bins')
    print(f"  {name} (4 bins): Acc={acc:.4f} F1={f1:.4f}")

  GB (4 bins): Acc=0.3617 F1=0.2993
  ExtraTrees (4 bins): Acc=0.3040 F1=0.3017
  SVM_linear (4 bins): Acc=0.3367 F1=0.2542


In [45]:
# --- HourBin3: Try 3 bins (Night 0-8, Day 8-16, Evening 16-24) ---
# Random baseline = 33.3%
def hour_to_3bin(h):
    if 0 <= h < 8: return 0   # Night
    if 8 <= h < 16: return 1  # Day
    return 2  # Evening

df['HourBin3'] = df['Hour'].apply(hour_to_3bin)
y3_3bin = df['HourBin3']

print("HourBin3 (3 classes): random baseline 33.3%. Running models...")
for name, mdl in [
    ('GB', GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('ExtraTrees', ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('SVM_linear', LinearSVC(max_iter=2000, random_state=42, dual='auto'))
]:
    X_tr, X_te, y_tr, y_te = train_test_split(X3_alt, y3_3bin, test_size=0.2, random_state=42, stratify=y3_3bin)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    mdl.fit(X_tr_s, y_tr)
    pred = mdl.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='weighted')
    log_experiment('HourBin3', f'{name}', {}, acc, f1, '3 bins')
    print(f"  {name} (3 bins): Acc={acc:.4f} F1={f1:.4f}")

HourBin3 (3 classes): random baseline 33.3%. Running models...
  GB (3 bins): Acc=0.4559 F1=0.4086
  ExtraTrees (3 bins): Acc=0.3905 F1=0.3876
  SVM_linear (3 bins): Acc=0.4228 F1=0.3663


In [46]:
# --- HourBin2: 2 bins (Night 0-12, Day 12-24) ---
# Random baseline = 50%
def hour_to_2bin(h):
    return 0 if 0 <= h < 12 else 1  # Night vs Day

df['HourBin2'] = df['Hour'].apply(hour_to_2bin)
y3_2bin = df['HourBin2']

print("HourBin2 (2 classes): random baseline 50%. Running models...")
for name, mdl in [
    ('GB', GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)),
    ('ExtraTrees', ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('SVM_linear', LinearSVC(max_iter=2000, random_state=42, dual='auto'))
]:
    X_tr, X_te, y_tr, y_te = train_test_split(X3_alt, y3_2bin, test_size=0.2, random_state=42, stratify=y3_2bin)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    mdl.fit(X_tr_s, y_tr)
    pred = mdl.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='weighted')
    log_experiment('HourBin2', f'{name}', {}, acc, f1, '2 bins')
    print(f"  {name} (2 bins): Acc={acc:.4f} F1={f1:.4f}")

HourBin2 (2 classes): random baseline 50%. Running models...
  GB (2 bins): Acc=0.6112 F1=0.4765
  ExtraTrees (2 bins): Acc=0.5599 F1=0.5539
  SVM_linear (2 bins): Acc=0.6113 F1=0.4638


## 11. Summary of New Experiments (Crime_Category & HourBin)

In [47]:
exp_all = pd.DataFrame(EXPERIMENT_LOG)
crime_exp = exp_all[exp_all['task'].isin(['Crime_Category', 'HourBin', 'HourBin4', 'HourBin3', 'HourBin2'])]
crime_best = crime_exp.loc[crime_exp.groupby('task')['accuracy'].idxmax()]
print("Best model per task (Crime_Category, HourBin, HourBin4, HourBin3, HourBin2):")
display(crime_best.sort_values('accuracy', ascending=False))

Best model per task (Crime_Category, HourBin, HourBin4, HourBin3, HourBin2):


,task,model,params,accuracy,f1_weighted,notes
36,HourBin2,SVM_linear,{},0.6113,0.4638,2 bins
22,Crime_Category,GB_alt_feat,{},0.5057,0.4562,Alt features
31,HourBin3,GB,{},0.4559,0.4086,3 bins
28,HourBin4,GB,{},0.3617,0.2993,4 bins
25,HourBin,GB_alt_feat,{},0.2525,0.2265,Alt features


## 6. Additional Hyperparameter Experiments (Extended Tuning)

Further tune to push toward 0.75+ accuracy.

In [ ]:
def extended_tune(X, y, task_name):
    class_counts = y.value_counts()
    valid = class_counts[class_counts >= 2].index
    m = y.isin(valid)
    Xf, yf = X[m].reset_index(drop=True), y[m].reset_index(drop=True)
    X_tr, X_te, y_tr, y_te = train_test_split(Xf, yf, test_size=0.2, random_state=42, stratify=yf)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    extended_grid = {
        'n_estimators': [150, 250, 300],
        'max_depth': [6, 8, 10, 12],
        'learning_rate': [0.03, 0.05, 0.1],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
    gs = GridSearchCV(GradientBoostingClassifier(random_state=42), extended_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1)
    gs.fit(X_tr_s, y_tr)
    pred = gs.predict(X_te_s)
    acc = accuracy_score(y_te, pred)
    f1 = f1_score(y_te, pred, average='weighted')
    log_experiment(task_name, 'GB_extended', gs.best_params_, acc, f1, 'Extended GridSearch')
    print(f"{task_name} Extended: Acc={acc:.4f} F1={f1:.4f}")
    print("Best params:", gs.best_params_)
    return gs

gs1_ext = extended_tune(X1, y1, 'Crime_Category')
gs2_ext = extended_tune(X2, y2, 'Beat')
gs3_ext = extended_tune(X3, y3, 'HourBin')

Fitting 3 folds for each of 144 candidates, totalling 432 fits


KeyboardInterrupt: 

## 7. Final Experiment Summary

In [ ]:
exp_df = pd.DataFrame(EXPERIMENT_LOG)
best_per_task = exp_df.loc[exp_df.groupby('task')['accuracy'].idxmax()]
print("Best model per task:")
display(best_per_task)
print("\nTarget: accuracy >= 0.75")
met = exp_df[exp_df['accuracy'] >= 0.75]
print(f"Experiments meeting target: {len(met)} / {len(exp_df)}")

# === LightGBM v2 Pipeline (Steps 1–8) ===

Improved pipeline: feature engineering, LightGBM with StratifiedKFold CV, class weights, and ceiling diagnostics.

In [ ]:
# === Step 2: Feature engineering function ===
def engineer_features(df):
    out = df.copy()
    # Interaction features (construct then label-encode)
    out['Beat_x_DayOfWeek'] = (out['Beat'].astype(str) + '_' + out['DayOfWeek'].astype(str))
    out['Beat_x_HourBin'] = (out['Beat'].astype(str) + '_' + out['HourBin'].astype(str))
    out['Crime_x_DayOfWeek'] = (out['Crime_Cat_enc'].astype(str) + '_' + out['DayOfWeek'].astype(str))
    out['LatLon_grid'] = (out['Lat_bin'].astype(str) + '_' + out['Lon_bin'].astype(str))
    out['Community_x_Month'] = (out['Community_Area'].astype(str) + '_' + out['Month'].astype(str))
    for col in ['Beat_x_DayOfWeek', 'Beat_x_HourBin', 'Crime_x_DayOfWeek', 'LatLon_grid', 'Community_x_Month']:
        out[col] = LabelEncoder().fit_transform(out[col].astype(str))

    # Target-encoded aggregate features (groupby transform)
    out['beat_crime_count'] = out.groupby('Beat')['Beat'].transform('count')
    out['beat_domestic_rate'] = out.groupby('Beat')['Domestic'].transform('mean')
    out['location_crime_count'] = out.groupby('Location_enc')['Location_enc'].transform('count')
    out['community_crime_count'] = out.groupby('Community_Area')['Community_Area'].transform('count')
    # beat_night_rate: 6-bin HourBin: 0=0-4, 5=20-24 are night
    out['beat_night_rate'] = out.groupby('Beat')['HourBin'].transform(
        lambda g: g.isin([0, 5]).mean()
    )
    return out

df = engineer_features(df)
print("engineer_features done. New columns added.")

engineer_features done. New columns added.


In [16]:
# === Step 3: Shared LightGBM runner ===
def run_lgbm_task(X, y_raw, task_name, lgb_params, n_folds=5, verbose=True):
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    n_classes = len(le.classes_)
    class_weights = len(y) / (n_classes * np.bincount(y, minlength=n_classes))
    sample_weights = class_weights[y]

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    accs, f1s = [], []
    for tr_idx, te_idx in skf.split(X, y):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]
        sw_tr = sample_weights[tr_idx]

        mdl = lgb.LGBMClassifier(**lgb_params)
        mdl.fit(
            X_tr, y_tr, sample_weight=sw_tr,
            eval_set=[(X_te, y_te)],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )
        pred = mdl.predict(X_te)
        accs.append(accuracy_score(y_te, pred))
        f1s.append(f1_score(y_te, pred, average='weighted', zero_division=0))

    mean_acc, std_acc = float(np.mean(accs)), float(np.std(accs))
    mean_f1, std_f1 = float(np.mean(f1s)), float(np.std(f1s))
    if verbose:
        print(f"{task_name}: Accuracy {mean_acc:.4f} ± {std_acc:.4f}, F1 (weighted) {mean_f1:.4f} ± {std_f1:.4f}")
    log_experiment(task_name, 'LightGBM_CV', lgb_params, mean_acc, mean_f1, 'StratifiedKFold CV')

    # Refit on full data (no eval_set, no early stopping)
    final_params = {**lgb_params}
    final_params.pop('callbacks', None)
    final_mdl = lgb.LGBMClassifier(**final_params)
    final_mdl.fit(X, y, sample_weight=sample_weights)
    return final_mdl, le

In [4]:
# === Step 7: Accuracy ceiling diagnostic (run before models) ===
def compute_baseline_and_ceiling(X, y, task_name):
    majority_acc = pd.Series(y).value_counts(normalize=True).iloc[0]
    if 'Beat' in X.columns and 'DayOfWeek' in X.columns:
        group_mode_acc = (
            pd.DataFrame({
                'y': np.array(y),
                'Beat': X['Beat'].values,
                'DayOfWeek': X['DayOfWeek'].values
            })
            .groupby(['Beat', 'DayOfWeek'])['y']
            .apply(lambda g: g.value_counts().iloc[0] / len(g))
            .mean()
        )
        print(f"\n{task_name}:")
        print(f"  Majority class baseline:          {majority_acc:.4f}")
        print(f"  Group-mode ceiling (Beat×Day):    {group_mode_acc:.4f}")
        print(f"  → Model < ceiling: more features can help")
        print(f"  → Model ≈ ceiling: inherent noise is the limit")

In [ ]:
# === Step 4: Task 1 — Crime Category ===
# 4a. Merge rare categories
cat_counts = df['Crime_Category'].value_counts(normalize=True)
rare_cats = cat_counts[cat_counts < 0.01].index.tolist()
df['Crime_Category_merged'] = df['Crime_Category'].apply(
    lambda x: 'OTHER' if x in rare_cats else x
)
print("Merged categories:", rare_cats)
print("\nCrime_Category_merged distribution:")
print(df['Crime_Category_merged'].value_counts(normalize=True))

feat_crime_v2 = [
    'Beat', 'Lat_bin', 'Lon_bin', 'HourBin', 'DayOfWeek', 'Month',
    'Location_enc', 'Domestic', 'Community_Area', 'IsWeekend', 'Quarter',
    'Beat_x_DayOfWeek', 'Beat_x_HourBin', 'LatLon_grid', 'Community_x_Month',
    'beat_crime_count', 'beat_domestic_rate', 'location_crime_count',
    'community_crime_count', 'beat_night_rate',
]
X1 = df[feat_crime_v2].astype(float)
y1 = df['Crime_Category_merged']
compute_baseline_and_ceiling(X1, y1, 'Crime_Category_v2')

crime_lgb_params = {
    'objective': 'multiclass',
    'num_class': y1.nunique(),
    'metric': 'multi_logloss',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'class_weight': 'balanced',
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1,
}
model_crime, le_crime = run_lgbm_task(X1, y1, 'Crime_Category_v2', crime_lgb_params)

Merged categories: []

Crime_Category_merged distribution:
Crime_Category_merged
Theft        0.328455
Violent      0.314235
Other        0.187585
Property     0.135095
Narcotics    0.034630
Name: proportion, dtype: float64

Crime_Category_v2:
  Majority class baseline:          0.3285
  Group-mode ceiling (Beat×Day):    0.4022
  → Model < ceiling: more features can help
  → Model ≈ ceiling: inherent noise is the limit
Crime_Category_v2: Accuracy 0.3072 ± 0.0017, F1 (weighted) 0.3351 ± 0.0020


In [18]:
# === Step 4d: Feature importance (Task 1) ===
imp = pd.DataFrame({
    'feature': feat_crime_v2,
    'importance': model_crime.feature_importances_
}).sort_values('importance', ascending=False)
print(imp.to_string(index=False))
top5 = imp.head(5)['feature'].tolist()
highlight = ['Beat_x_HourBin', 'beat_crime_count', 'beat_night_rate']
in_top5 = [f for f in highlight if f in top5]
print(f"\nNew interaction features in top 5: {in_top5 if in_top5 else 'None'}")

              feature  importance
                Month       15077
    Community_x_Month       13244
         Location_enc       12743
            DayOfWeek       12492
              HourBin       11286
 location_crime_count       11089
     beat_crime_count        9529
   beat_domestic_rate        9338
          LatLon_grid        9280
      beat_night_rate        9197
     Beat_x_DayOfWeek        7869
                 Beat        6835
       Beat_x_HourBin        6233
              Lon_bin        5462
community_crime_count        5297
       Community_Area        3220
              Lat_bin        2289
             Domestic        2123
              Quarter        1616
            IsWeekend         781

New interaction features in top 5: None


In [ ]:
# === Step 5: Task 3 — HourBin (4-bin) ===
if 'HourBin4' not in df.columns:
    def hour_to_4bin(h):
        if 0 <= h < 6:   return 0  # night
        if 6 <= h < 12:  return 1  # morning
        if 12 <= h < 18: return 2  # afternoon
        return 3  # evening
    df['HourBin4'] = df['Hour'].apply(hour_to_4bin)

feat_hour_v2 = [
    'Beat', 'Crime_Cat_enc', 'DayOfWeek', 'Month', 'Location_enc',
    'IsWeekend', 'Lat_bin', 'Lon_bin', 'Community_Area',
    'Beat_x_DayOfWeek', 'Crime_x_DayOfWeek', 'LatLon_grid',
    'beat_night_rate', 'location_crime_count', 'community_crime_count',
]
X3 = df[feat_hour_v2].astype(float)
y3 = df['HourBin4']
print("Random baseline (4 classes): 0.25")
compute_baseline_and_ceiling(X3, y3, 'HourBin4_v2')

hour_lgb_params = {
    'objective': 'multiclass',
    'num_class': 4,
    'metric': 'multi_logloss',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'class_weight': 'balanced',
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1,
}
model_hour, le_hour = run_lgbm_task(X3, y3, 'HourBin4_v2', hour_lgb_params)

Random baseline (4 classes): 0.25

HourBin4_v2:
  Majority class baseline:          0.3156
  Group-mode ceiling (Beat×Day):    0.3498
  → Model < ceiling: more features can help
  → Model ≈ ceiling: inherent noise is the limit
HourBin4_v2: Accuracy 0.2630 ± 0.0020, F1 (weighted) 0.2159 ± 0.0044


In [ ]:
# === Step 6: Task 2 — Beat Prediction (top 50, single split) ===
top_beats = df['Beat'].value_counts().head(50).index.tolist()
mask_b = df['Beat'].isin(top_beats)
feat_beat_v2 = [
    'Lat_bin', 'Lon_bin', 'Crime_Cat_enc', 'HourBin', 'DayOfWeek', 'Month',
    'Location_enc', 'Community_Area', 'IsWeekend',
    'LatLon_grid', 'Beat_x_DayOfWeek', 'Community_x_Month',
    'beat_crime_count', 'beat_domestic_rate',
]
X2 = df.loc[mask_b, feat_beat_v2].astype(float)
y2 = df.loc[mask_b, 'Beat'].astype(str)

X2_tr, X2_te, y2_tr, y2_te = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)
le_beat = LabelEncoder()
y2_tr_enc = le_beat.fit_transform(y2_tr)
y2_te_enc = le_beat.transform(y2_te)
n_cl = len(le_beat.classes_)
cw = len(y2_tr_enc) / (n_cl * np.bincount(y2_tr_enc, minlength=n_cl))
sw = cw[y2_tr_enc]

beat_lgb_params = {
    'objective': 'multiclass',
    'num_class': n_cl,
    'metric': 'multi_logloss',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_child_samples': 10,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'class_weight': 'balanced',
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1,
}
model_beat = lgb.LGBMClassifier(**beat_lgb_params)
model_beat.fit(X2_tr, y2_tr_enc, sample_weight=sw, callbacks=[lgb.log_evaluation(-1)])
pred_beat = model_beat.predict(X2_te)
acc_beat = accuracy_score(y2_te_enc, pred_beat)
f1_beat = f1_score(y2_te_enc, pred_beat, average='weighted', zero_division=0)
print(f"Beat_v2: Accuracy {acc_beat:.4f}, F1 (weighted) {f1_beat:.4f}")
log_experiment('Beat_v2', 'LightGBM_single_split', beat_lgb_params, acc_beat, f1_beat, 'Top 50 beats, 80/20 split')

Beat_v2: Accuracy 1.0000, F1 (weighted) 1.0000


In [21]:
# === Step 8: Final comparison table ===
exp_df = pd.DataFrame(EXPERIMENT_LOG)
print("\nAll experiments:")
print(exp_df.sort_values(['task', 'accuracy'], ascending=[True, False]).to_string(index=False))

print("\nBest model per task:")
best = exp_df.loc[exp_df.groupby('task')['accuracy'].idxmax()]
print(best[['task', 'model', 'accuracy', 'f1_weighted', 'notes']].to_string(index=False))


All experiments:
             task                 model                                                                                                                                                                                                                                                                                                                           params  accuracy  f1_weighted                     notes
          Beat_v2 LightGBM_single_split                {'objective': 'multiclass', 'num_class': 50, 'metric': 'multi_logloss', 'n_estimators': 500, 'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 10, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'class_weight': 'balanced', 'verbose': -1, 'random_state': 42, 'n_jobs': -1}    1.0000       1.0000 Top 50 beats, 80/20 split
Crime_Category_v2           LightGBM_CV {'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss', 'n_estimators': 500, 'learning_rate': 0.05, 'num_le